# 10.04 -- Tree Decoding: Unified Framework for Speculative Methods

**Module:** 10 -- Advanced Inference  
**Notebook:** 4 of 5  
**Prerequisites:** 10.01 (Speculative Decoding), 10.02 (EAGLE), 10.03 (Medusa)

---

## Learning Objectives

1. Understand tree decoding as the unified framework underlying EAGLE, Medusa, and SpecInfer
2. Implement a draft tree data structure with branch pruning
3. Implement tree attention from scratch: the custom mask that verifies all branches in parallel
4. Implement the tree-level acceptance algorithm (a generalisation of speculative sampling)
5. Measure how tree width and depth trade off against speedup and memory

---

## 1. Why Tree Decoding

The speculative methods in the previous two notebooks all face the same verification bottleneck:
they generate multiple candidate token sequences, then must verify each against the target model.

Naive verification runs one target model forward pass per candidate, costing as much compute
as generating that many tokens autoreggressively -- negating the benefit.

**Tree decoding** solves this by recognising that candidate sequences always share a common
prefix (the original prompt plus any accepted tokens so far). By arranging the candidates as
branches of a tree and using a custom attention mask, the target model can verify all branches
in a single forward pass, paying only once for their shared prefix.

```
Draft tree (depth=2, branching=2, prompt=[A, B, C]):

              [A, B, C]
             /         \
          [D]           [E]
         /   \         /   \
       [D,F] [D,G]  [E,H] [E,I]

Target model forward pass sees: A B C D F   <- branch 1
                                A B C D G   <- branch 2   but only computed once
                                A B C E H   <- branch 3   for shared prefixes
                                A B C E I   <- branch 4

Tree attention mask ensures each branch only attends to its own ancestors.
```

Tree decoding is not a new draft method -- it is an efficient verification strategy
that any draft method (EAGLE, Medusa, independent model) can use.


In [ ]:
# Environment setup.
#
# Tree decoding is primarily a data structure and masking problem.
# We implement the full tree from scratch: node representation, tree construction,
# tree attention mask generation, and tree-level acceptance.
# No large models are required; we demonstrate on GPT-2.

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')


## 2. Draft Tree Data Structure


In [ ]:
# Draft tree data structure.
#
# A DraftTree is a k-ary tree where:
#   - The root is a virtual node representing the end of the prompt.
#   - Each non-root node holds one draft token and its probability.
#   - The children of a node are the top-k candidates for the next position.
#   - A path from root to a leaf is one complete candidate sequence.
#
# We index nodes by their position in a flattened list (breadth-first order).
# This flat representation is what gets passed to the attention mechanism:
# the model sees a sequence of tokens, and the attention mask encodes the
# tree structure by controlling which tokens can attend to which others.
#
# The parent_indices list is the core data structure:
#   parent_indices[i] = j  means node i is a child of node j.
# Node 0 is the root (parent = -1).
# This list is used to construct the attention mask row by row.

@dataclass
class TreeNode:
    """One node in the draft tree."""
    node_id:      int
    token_id:     int
    token_prob:   float
    parent_id:    int           # -1 for root
    depth:        int
    children:     List[int] = field(default_factory=list)

    @property
    def is_root(self): return self.parent_id == -1


class DraftTree:
    """
    A tree of draft token candidates.

    Constructed by repeatedly expanding each leaf node with its top-k
    candidate next tokens according to a draft distribution.

    The tree is stored as a flat list of nodes in breadth-first order.
    Node 0 is always the root (a placeholder, not a real token).
    """

    def __init__(self):
        # Root node: represents the end of the prompt, no token
        root = TreeNode(node_id=0, token_id=-1, token_prob=1.0,
                        parent_id=-1, depth=0)
        self.nodes: List[TreeNode] = [root]

    def expand(
        self,
        parent_id:    int,
        candidates:   List[Tuple[int, float]],   # (token_id, probability)
    ) -> List[int]:
        """Add children to a given node. Returns the new node ids."""
        parent     = self.nodes[parent_id]
        new_ids    = []
        for tok_id, prob in candidates:
            nid = len(self.nodes)
            node = TreeNode(
                node_id   = nid,
                token_id  = tok_id,
                token_prob = prob,
                parent_id  = parent_id,
                depth      = parent.depth + 1,
            )
            self.nodes.append(node)
            parent.children.append(nid)
            new_ids.append(nid)
        return new_ids

    def leaves(self) -> List[int]:
        """Return node ids of all leaf nodes (no children)."""
        return [n.node_id for n in self.nodes if not n.children]

    def path_to_root(self, node_id: int) -> List[int]:
        """Return the path from node_id up to (but not including) the root."""
        path = []
        nid  = node_id
        while nid != 0:
            path.append(nid)
            nid = self.nodes[nid].parent_id
        return list(reversed(path))

    def all_sequences(self) -> List[List[int]]:
        """Return all root-to-leaf token sequences."""
        seqs = []
        for leaf in self.leaves():
            path = self.path_to_root(leaf)
            seqs.append([self.nodes[nid].token_id for nid in path])
        return seqs

    def ancestor_mask(self) -> torch.Tensor:
        """
        Build the tree attention mask for the non-root nodes.

        Returns a boolean mask M of shape (N-1, N-1) where N = len(self.nodes).
        M[i, j] = True means node i+1 can attend to node j+1.

        Node i can attend to node j iff j is an ancestor of i
        (i.e., j lies on the path from root to i).
        This is the defining property of tree attention.
        """
        non_root = self.nodes[1:]    # skip the root placeholder
        N        = len(non_root)
        mask     = torch.zeros(N, N, dtype=torch.bool)

        for i, node in enumerate(non_root):
            # Collect ancestors of this node (in non-root index space)
            anc_nid = node.node_id
            while anc_nid != 0:
                mask[i, anc_nid - 1] = True   # can attend to self and ancestors
                anc_nid = self.nodes[anc_nid].parent_id

        return mask   # True = allowed to attend


# Build a small example tree: branching=2, depth=2
tree = DraftTree()

# Depth 1: two candidates after the prompt
depth1 = tree.expand(parent_id=0, candidates=[(42, 0.45), (17, 0.30)])

# Depth 2: two candidates for each depth-1 node
for nid in depth1:
    tree.expand(parent_id=nid, candidates=[(88, 0.35), (55, 0.22)])

print(f'Tree: {len(tree.nodes)} nodes (1 root + {len(tree.nodes)-1} draft nodes)')
print(f'Leaf nodes: {tree.leaves()}')
print(f'Candidate sequences:')
for seq in tree.all_sequences():
    print(f'  {seq}')
print()

mask = tree.ancestor_mask()
print(f'Tree attention mask ({mask.shape[0]}x{mask.shape[1]}):')
for row in mask:
    print(' ', ''.join(['1' if v else '0' for v in row]))


## 3. Tree Attention Mask


In [ ]:
# Tree attention mask visualisation and analysis.
#
# The tree attention mask is the core of efficient multi-candidate verification.
# It encodes the tree structure as a binary matrix: entry (i, j) = 1 means
# node i is allowed to attend to node j during self-attention.
#
# Compare with the two alternatives:
#
#   Causal mask (standard autoregressive):
#     Lower-triangular: each position attends to all previous positions.
#     All tokens share the same context -- no branching.
#
#   Independent mask (naive multi-candidate verification):
#     Block-diagonal: each candidate sequence is isolated from all others.
#     Equivalent to running separate forward passes per candidate.
#     Correct, but wastes compute by not sharing the common prefix.
#
#   Tree mask (tree attention):
#     Each node attends to its own ancestors in the tree.
#     Nodes in different branches cannot see each other.
#     Nodes sharing a prefix share the computation for that prefix.
#     This is strictly more efficient than independent mask.

def build_full_mask(
    prompt_len:  int,
    tree:        DraftTree,
    full_seq_len: int,
) -> torch.Tensor:
    """
    Build the complete attention mask for (prompt + draft tree) sequence.

    The full sequence is: [prompt tokens ... | draft node 1 | draft node 2 | ...]

    All draft nodes can attend to all prompt tokens (shared prefix).
    Among themselves, draft nodes follow the tree attention pattern.
    """
    n_draft = len(tree.nodes) - 1   # exclude root
    total   = prompt_len + n_draft
    mask    = torch.zeros(total, total, dtype=torch.bool)

    # Prompt tokens: standard causal attention (lower-triangular)
    for i in range(prompt_len):
        mask[i, :i + 1] = True

    # Draft nodes: attend to all prompt tokens
    for d in range(n_draft):
        mask[prompt_len + d, :prompt_len] = True

    # Draft nodes: attend to tree ancestors
    tree_mask = tree.ancestor_mask()   # (n_draft, n_draft)
    mask[prompt_len:, prompt_len:] = tree_mask

    return mask


# Visualise three mask types for a small example
PROMPT_LEN = 4
full_mask  = build_full_mask(PROMPT_LEN, tree, PROMPT_LEN + len(tree.nodes) - 1)

# Standard causal mask (no branching)
n_total   = full_mask.shape[0]
causal    = torch.tril(torch.ones(n_total, n_total, dtype=torch.bool))

# Independent mask: block-diagonal for draft tokens
indep = torch.zeros_like(full_mask)
indep[:PROMPT_LEN, :PROMPT_LEN] = torch.tril(torch.ones(PROMPT_LEN, PROMPT_LEN, dtype=torch.bool))
n_draft  = len(tree.nodes) - 1
n_leaves = len(tree.leaves())
leaf_seqs = tree.all_sequences()
for seq in leaf_seqs:
    for j_idx, nid in enumerate(seq):
        row = PROMPT_LEN + nid - 1
        indep[row, :PROMPT_LEN] = True
        for k in range(j_idx + 1):
            col = PROMPT_LEN + seq[k] - 1
            indep[row, col] = True


fig, axes = plt.subplots(1, 3, figsize=(15, 5))

mask_names = ['Causal (no branching)', 'Tree Attention', 'Independent (naive)']
masks_list = [causal, full_mask, indep]

for ax, name, m in zip(axes, mask_names, masks_list):
    ax.imshow(m.float().numpy(), cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.axvline(PROMPT_LEN - 0.5, color='orange', lw=2, label='Prompt/draft boundary')
    ax.axhline(PROMPT_LEN - 0.5, color='orange', lw=2)
    ax.set_title(f'{name}\nActive entries: {m.sum().item()}', fontsize=11)
    ax.set_xlabel('Key position', fontsize=9)
    ax.set_ylabel('Query position', fontsize=9)
    ax.set_xticks(range(n_total))
    ax.set_yticks(range(n_total))
    xlbls = [f'P{i}' for i in range(PROMPT_LEN)] + [f'D{i}' for i in range(1, n_draft+1)]
    ax.set_xticklabels(xlbls, fontsize=7, rotation=45)
    ax.set_yticklabels(xlbls, fontsize=7)

axes[1].legend(handles=[
    plt.Rectangle((0,0), 1, 1, color=plt.cm.Blues(0.8), label='Can attend'),
    plt.Rectangle((0,0), 1, 1, color='white', ec='gray', label='Cannot attend'),
], loc='upper right', fontsize=8)

plt.suptitle('Attention Mask Comparison for Draft Tree Verification', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_tree_mask.png', dpi=150, bbox_inches='tight')
plt.show()

print('Active attention entries:')
print(f'  Causal:      {causal.sum().item()}')
print(f'  Tree attn:   {full_mask.sum().item()}  <- reuses shared prefix computation')
print(f'  Independent: {indep.sum().item()}  <- wastes compute duplicating prompt attention')


## 4. Tree-Level Acceptance


In [ ]:
# Tree-level speculative acceptance algorithm.
#
# After the target model runs with tree attention, we have target probabilities
# at every node in the tree. We now want to find the longest prefix of any
# root-to-leaf path that the target model would have generated with high probability.
#
# The tree acceptance algorithm proceeds depth-by-depth:
#   1. At depth d, try to accept each node at that depth using the standard
#      speculative acceptance criterion: accept node n with probability
#      min(1, p_target(n.token | ancestors) / p_draft(n.token | ancestors)).
#   2. If a node is accepted, its children become candidates at depth d+1.
#   3. If rejected, prune the subtree rooted at that node.
#   4. Stop at the deepest accepted node; if multiple nodes at that depth
#      are accepted, take the one with the highest cumulative probability.
#
# This generalises the standard linear speculative acceptance to tree-structured
# draft sequences. The resulting token sequence is always a valid sample from
# the target distribution (the same lossless guarantee as flat speculative decoding).

def tree_acceptance(
    tree:          DraftTree,
    draft_probs:   Dict[int, torch.Tensor],   # node_id --> (vocab,) draft probs
    target_probs:  Dict[int, torch.Tensor],   # node_id --> (vocab,) target probs
) -> List[int]:
    """
    Run the tree-level speculative acceptance algorithm.

    Returns the accepted token sequence (path from root toward a leaf,
    stopping at first rejection or reaching a leaf).

    The returned sequence is a valid sample from the target distribution.
    """
    accepted_path    = []     # accepted node ids
    current_frontier = tree.nodes[0].children   # start with root children

    while current_frontier:
        # Try to accept each node in the current frontier
        accepted_at_depth = []

        for nid in current_frontier:
            node  = tree.nodes[nid]
            tok   = node.token_id
            p_d   = draft_probs.get(nid, None)
            p_t   = target_probs.get(nid, None)

            if p_d is None or p_t is None:
                continue

            # Acceptance probability: min(1, p_target / p_draft)
            ratio    = (p_t[tok] / (p_d[tok] + 1e-10)).item()
            accepted = (torch.rand(1).item() < min(1.0, ratio))

            if accepted:
                # Track cumulative path probability for tie-breaking
                cum_prob = node.token_prob
                accepted_at_depth.append((nid, cum_prob))

        if not accepted_at_depth:
            # No node accepted at this depth: stop and resample from target
            # (the resampling step is omitted here for clarity)
            break

        # Take the accepted node with highest cumulative probability
        best_nid = max(accepted_at_depth, key=lambda x: x[1])[0]
        accepted_path.append(best_nid)

        # Next frontier: children of the accepted node
        current_frontier = tree.nodes[best_nid].children

    return [tree.nodes[nid].token_id for nid in accepted_path]


# Simulate tree acceptance with mock probabilities
torch.manual_seed(0)
vocab_size = 1000

# Assign random but plausible draft/target probs to each tree node
mock_draft  = {}
mock_target = {}
for node in tree.nodes[1:]:
    p_d = torch.zeros(vocab_size); p_d[node.token_id % vocab_size] = 0.4
    p_d += torch.rand(vocab_size) * 0.01; p_d /= p_d.sum()
    mock_draft[node.node_id] = p_d

    p_t = torch.zeros(vocab_size); p_t[node.token_id % vocab_size] = 0.35
    p_t += torch.rand(vocab_size) * 0.01; p_t /= p_t.sum()
    mock_target[node.node_id] = p_t

n_trials = 200
accepted_lens = []
for _ in range(n_trials):
    result = tree_acceptance(tree, mock_draft, mock_target)
    accepted_lens.append(len(result))

print(f'Tree acceptance over {n_trials} trials:')
print(f'  Mean accepted tokens:  {np.mean(accepted_lens):.2f}')
print(f'  Fraction accepting 0:  {accepted_lens.count(0)/n_trials:.2f}')
print(f'  Fraction accepting 1:  {accepted_lens.count(1)/n_trials:.2f}')
print(f'  Fraction accepting 2:  {accepted_lens.count(2)/n_trials:.2f}')
print(f'  Max possible:          {max(len(seq) for seq in tree.all_sequences())}')


In [ ]:
# Tree width and depth trade-off analysis.
#
# A wider, deeper tree generates more candidates and potentially accepts more tokens
# per target model call. However:
#
#   - Wider trees require more KV cache memory (one slot per tree node).
#   - Deeper trees require more draft model compute.
#   - Verification cost scales with tree size, not just accepted depth.
#
# The optimal tree shape maximises tokens accepted per unit of total compute.
# Empirically, asymmetric trees (wider at shallow depths, narrower at deeper depths)
# outperform symmetric trees because acceptance rate drops with depth.
#
# The figure below sweeps tree configurations and shows the compute-acceptance
# trade-off. The annotations show the (branching, depth) configuration at each point.

configs = [
    (1, 1), (2, 1), (3, 1), (4, 1),
    (1, 2), (2, 2), (3, 2),
    (1, 3), (2, 3),
    (1, 4), (2, 4),
]

# Analytical model:
#   - acceptance rate at depth d: alpha^d  (alpha = 0.80 for trained drafter)
#   - expected accepted tokens = sum_{d=1}^{D} alpha^d * P(reach depth d)
#   - tree size = (k^{D+1} - 1) / (k - 1)  for branching k, depth D
alpha = 0.80

results_tree = []
for k, D in configs:
    # Tree size (nodes)
    if k == 1:
        tree_size = D
    else:
        tree_size = (k**(D+1) - 1) // (k - 1) - 1   # exclude root

    # Expected accepted tokens
    expected_accepted = sum(alpha**d for d in range(1, D+1))

    # Effective speedup: tokens accepted per target call, accounting for
    # verification cost proportional to tree_size
    # Cost model: each tree node costs ~0.1 relative to one autoregressive step
    verification_overhead = tree_size * 0.10
    net_speedup = (1 + expected_accepted) / (1 + verification_overhead)

    results_tree.append({
        'config': f'k={k},D={D}',
        'tree_size': tree_size,
        'expected_accepted': expected_accepted,
        'net_speedup': net_speedup,
    })


fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.40, wspace=0.32)

# Panel 1: tree size vs expected accepted tokens
ax = fig.add_subplot(gs[0, 0])
tree_sizes = [r['tree_size'] for r in results_tree]
expected   = [r['expected_accepted'] for r in results_tree]
sc = ax.scatter(tree_sizes, expected, c=[r['net_speedup'] for r in results_tree],
                cmap='RdYlGn', s=120, zorder=5, vmin=0.9, vmax=2.5)
plt.colorbar(sc, ax=ax, label='Net speedup')
for r in results_tree:
    ax.annotate(r['config'], (r['tree_size'], r['expected_accepted']),
                xytext=(3, 3), textcoords='offset points', fontsize=7)
ax.set_xlabel('Tree size (nodes)', fontsize=11)
ax.set_ylabel('Expected accepted tokens', fontsize=11)
ax.set_title(f'Tree Size vs Expected Accepted\n(alpha={alpha}, colour=net speedup)', fontsize=12)
ax.grid(alpha=0.3)

# Panel 2: net speedup by configuration
ax = fig.add_subplot(gs[0, 1])
configs_sorted = sorted(results_tree, key=lambda r: -r['net_speedup'])
labels  = [r['config'] for r in configs_sorted[:8]]
speeds  = [r['net_speedup'] for r in configs_sorted[:8]]
bars = ax.barh(range(len(labels)), speeds, color='#2ca02c', alpha=0.85)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Net speedup (accounting for verification cost)', fontsize=11)
ax.set_title('Top Configurations by Net Speedup', fontsize=12)
for bar, v in zip(bars, speeds):
    ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
            f'{v:.2f}x', va='center', fontsize=9)
ax.grid(alpha=0.3, axis='x')

# Panel 3: acceptance rate by depth for different alpha values
ax = fig.add_subplot(gs[1, 0])
depths_range = range(1, 8)
for al, col in [(0.95,'#2ca02c'), (0.80,'#ff7f0e'), (0.65,'#d62728')]:
    cumprob = [al**d for d in depths_range]
    ax.plot(list(depths_range), cumprob, 'o-', lw=2, ms=6, color=col,
            label=f'alpha={al} (per-token accept rate)')
ax.set_xlabel('Tree depth', fontsize=11)
ax.set_ylabel('P(accepted at this depth)', fontsize=11)
ax.set_title('Acceptance Probability vs Depth\nfor Different Per-Token Acceptance Rates', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Panel 4: memory vs speedup trade-off
ax = fig.add_subplot(gs[1, 1])
memory_per_node_mb = 0.5   # approximate KV cache per node, small model
for r in results_tree:
    mem = r['tree_size'] * memory_per_node_mb
    ax.scatter(mem, r['net_speedup'], s=80, color='#1f77b4', alpha=0.7)
    ax.annotate(r['config'], (mem, r['net_speedup']),
                xytext=(2, 2), textcoords='offset points', fontsize=7)
ax.set_xlabel('KV cache memory for draft tree (MB, illustrative)', fontsize=11)
ax.set_ylabel('Net speedup', fontsize=11)
ax.set_title('Memory vs Speedup Trade-off\n(Each point is one tree configuration)', fontsize=12)
ax.grid(alpha=0.3)

plt.suptitle('Tree Decoding: Width/Depth Trade-off Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/10_tree_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

best = max(results_tree, key=lambda r: r['net_speedup'])
print(f'Best configuration: {best["config"]}  net speedup={best["net_speedup"]:.2f}x')


## 5. Engineering Notes

**Tree attention in practice**

Implementing tree attention requires modifying the model's attention forward pass to
accept a custom boolean mask. In HuggingFace models, this is done by passing the mask
to `attention_mask` with appropriate shape. vLLM has native tree attention support;
other frameworks require patching the attention kernel.

**Draft method -- tree shape pairing**

| Draft method | Natural tree shape | Reason |
|--------------|-------------------|--------|
| Independent small model | Wide, shallow (k=3-5, D=1-2) | Low acceptance at depth > 2 |
| EAGLE | Narrow, deep (k=2, D=4-6) | High acceptance enables deep trees |
| Medusa | Wide, medium (k=2-3, D=3-4) | Multiple heads naturally form wide trees |
| Lookup (prompt caching) | Very wide, D=1 | Exact matches or nothing |

**SpecInfer** (Miao et al., 2023) is the paper that first formalised tree decoding as
a general framework and proved its correctness. It showed that any tree-structured
acceptance algorithm that reduces to standard speculative acceptance along each path
is lossless (produces samples from the target distribution).

## 6. Exercises

1. **Correctness verification**: Run `tree_acceptance` with `mock_target == mock_draft`
   (perfect drafter). Verify that all leaf paths are accepted 100 percent of the time.
   Then set mock_target to be uniform (random model). Verify that acceptance rate
   drops to approximately 1 / vocab_size.

2. **Asymmetric tree**: Implement a tree where each node at depth 1 has k=3 children
   but each node at depth 2 has only k=1 child (greedy). Compare the expected accepted
   tokens against a symmetric k=2, D=2 tree analytically.

3. **Tree attention implementation**: Given the `full_mask` built above, pass it to a
   GPT-2 forward call via `attention_mask`. Verify that the logits at each draft node
   match those you would get from a separate forward pass over just the ancestors of
   that node.

4. **Optimal tree search**: For a given acceptance rate alpha and draft compute budget
   B nodes, find the branching/depth combination that maximises expected accepted tokens.
   Is the optimal tree symmetric or asymmetric?

5. **Real tree decoding**: Combine the EAGLE drafter from notebook 10.02 with the tree
   acceptance algorithm above. Use EAGLE to populate a k=2, D=3 tree. Run tree
   attention with GPT-2, then accept. Measure wall-clock tokens/second vs
   autoregressive baseline.

---

## 7. References

- [Miao et al. (2023) -- SpecInfer: Accelerating LLM Serving with Tree-based Speculative Inference and Verification](https://arxiv.org/abs/2305.09781)
- [Cai et al. (2024) -- Medusa: Simple LLM Inference Acceleration with Multiple Decoding Heads](https://arxiv.org/abs/2401.10774)
- [Li et al. (2024) -- EAGLE-2: Faster Inference with Dynamic Draft Trees](https://arxiv.org/abs/2406.16858)
- [Leviathan et al. (2023) -- Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
